<a href="https://colab.research.google.com/github/YangEonPil/Git-Practice/blob/main/make_regular_expression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import re
import math
import unicodedata
from dataclasses import dataclass
from typing import List, Optional, Dict, Any

import pandas as pd


# ============================================================
# 1. 기본 전처리 함수
# ============================================================

def normalize_text(text: str) -> str:
    """
    유튜브 자막 / 광고 문장 전처리.
    - 유니코드 정규화
    - 소문자화
    - 공백 정리
    """
    if text is None:
        return ""

    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def is_empty(value) -> bool:
    """
    엑셀 셀 값이 비어 있는지 확인.
    """
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if str(value).strip() == "":
        return True
    return False


def split_keywords(cell_value) -> List[str]:
    """
    엑셀의 '키워드사전1,2,3' 셀을 리스트로 변환.
    쉼표 기준 분리.
    """
    if is_empty(cell_value):
        return []

    text = str(cell_value).strip()

    # DB 검증 메모 같은 항목은 키워드로 쓰지 않음
    if text.startswith("(+"):
        return []

    keywords = [kw.strip() for kw in text.split(",")]
    keywords = [kw for kw in keywords if kw]

    return keywords


# ============================================================
# 2. 키워드 → 정규표현식 변환 함수
# ============================================================

def keyword_to_regex(keyword: str) -> str:
    """
    하나의 키워드를 정규표현식 안전 문자열로 변환.

    예:
    '저급 원료' → '저급\\s*원료'
    'No.1' → 'no\\.1'
    'before after' → 'before\\s*after'
    """
    keyword = normalize_text(keyword)

    # 괄호 메모 제거
    keyword = keyword.strip()
    keyword = re.sub(r"^\(|\)$", "", keyword)

    # 공백이 있는 표현은 공백 유무를 허용
    parts = keyword.split()
    escaped_parts = [re.escape(part) for part in parts]

    return r"\s*".join(escaped_parts)


def make_alt_pattern(keywords: List[str]) -> str:
    """
    키워드 리스트를 OR 정규표현식으로 변환.

    예:
    ['당뇨', '고혈압', '관절염']
    → '(?:고혈압|관절염|당뇨)'
    """
    cleaned = []

    for kw in keywords:
        kw = normalize_text(kw)
        if not kw:
            continue

        # '(생략가능 : ...)' 같은 설명은 처리
        kw = kw.replace("(생략가능 :", "")
        kw = kw.replace(")", "")
        kw = kw.strip()

        if not kw:
            continue

        cleaned.append(kw)

    # 긴 표현을 먼저 매칭해야 짧은 표현이 먼저 잡히는 문제를 줄일 수 있음
    cleaned = sorted(set(cleaned), key=len, reverse=True)

    if not cleaned:
        return ""

    return "(?:" + "|".join(keyword_to_regex(kw) for kw in cleaned) + ")"


def make_single_pattern(keywords: List[str]) -> Optional[str]:
    """
    단독 키워드형 패턴.
    """
    alt = make_alt_pattern(keywords)

    if not alt:
        return None

    return alt


def make_all_groups_pattern(keyword_groups: List[List[str]]) -> Optional[str]:
    """
    조합형 패턴.
    같은 문장 안에 각 키워드 그룹이 모두 등장해야 함.

    예:
    질병 + 치료표현
    → (?=.*(?:당뇨|고혈압))(?=.*(?:치료|완치|개선))
    """
    lookaheads = []

    for group in keyword_groups:
        alt = make_alt_pattern(group)
        if alt:
            lookaheads.append(f"(?=.*{alt})")

    if not lookaheads:
        return None

    return "".join(lookaheads)


def make_ordered_near_pattern(
    group1: List[str],
    group2: List[str],
    group3: Optional[List[str]] = None,
    max_gap: int = 12
) -> Optional[str]:
    """
    순서가 어느 정도 중요한 경우 사용할 수 있는 근접 패턴.

    예:
    항 + 암 + 효과
    → 항.{0,12}암.{0,12}효과

    단, 기본 구현에서는 lookahead를 주로 쓰고,
    항 효과형 / 한약처방형 같은 특수 케이스에 사용한다.
    """
    alt1 = make_alt_pattern(group1)
    alt2 = make_alt_pattern(group2)

    if not alt1 or not alt2:
        return None

    if group3:
        alt3 = make_alt_pattern(group3)
        if not alt3:
            return None
        return rf"{alt1}.{{0,{max_gap}}}{alt2}.{{0,{max_gap}}}{alt3}"

    return rf"{alt1}.{{0,{max_gap}}}{alt2}"


# ============================================================
# 3. Rule 객체
# ============================================================

@dataclass
class RulePattern:
    rule_id: str
    category: str
    subtype: str
    combination: str
    pattern: Optional[str]
    weight: int
    requires_db_check: bool = False
    exception_type: Optional[str] = None


# ============================================================
# 4. 세부 유형별 패턴 생성 로직
# ============================================================

def generate_pattern_for_row(row: pd.Series) -> RulePattern:
    """
    스프레드시트 한 행을 받아 세부 유형별 정규표현식을 생성.
    """

    category = "" if is_empty(row.get("유형")) else str(row.get("유형")).strip()
    subtype = "" if is_empty(row.get("세부 유형")) else str(row.get("세부 유형")).strip()
    combination = "" if is_empty(row.get("가능한 조합")) else str(row.get("가능한 조합")).strip()

    k1 = split_keywords(row.get("키워드사전1"))
    k2 = split_keywords(row.get("키워드사전2"))
    k3 = split_keywords(row.get("키워드사전3"))

    weight_value = row.get("위험도 가중치")
    weight = 0 if is_empty(weight_value) else int(weight_value)

    # 유형 번호 생성
    # 예: category = "1.1 질병(증상) 예방 치료", subtype = "A. 질병/증상 예방·치료형"
    category_id_match = re.search(r"\d+\.\d+", category)
    category_id = category_id_match.group(0) if category_id_match else ""

    subtype_letter_match = re.search(r"^[A-Z]", subtype)
    subtype_letter = subtype_letter_match.group(0) if subtype_letter_match else ""

    rule_id = f"{category_id}{subtype_letter}" if category_id and subtype_letter else subtype

    requires_db_check = "DB" in combination or "db" in combination.lower()

    pattern = None
    exception_type = None

    # ------------------------------------------------------------
    # 예외처리 행
    # ------------------------------------------------------------
    if "예외처리" in subtype:
        exception_type = combination
        return RulePattern(
            rule_id=rule_id,
            category=category,
            subtype=subtype,
            combination=combination,
            pattern=None,
            weight=0,
            requires_db_check=False,
            exception_type=exception_type
        )

    # ------------------------------------------------------------
    # 1.1B 항 효과형: 항 + 질병 + 효과표현
    # ------------------------------------------------------------
    if "항표현" in combination:
        pattern = make_ordered_near_pattern(k1, k2, k3, max_gap=8)

    # ------------------------------------------------------------
    # 1.3B 한약처방 조합형: 한약처방어근 + 한약제형명
    # 붙어서 나오는 경우가 많으므로 근접 패턴 사용
    # ------------------------------------------------------------
    elif "한약처방어근" in combination:
        pattern = make_ordered_near_pattern(k1, k2, None, max_gap=2)

    # ------------------------------------------------------------
    # 3.1A 최초/최고/유일/1위
    # 범위표현은 생략 가능.
    # 따라서 키워드사전2 + 키워드사전3을 필수 조건으로 둠.
    # 키워드사전1은 있으면 좋지만 필수 아님.
    # ------------------------------------------------------------
    elif "범위표현" in combination and "최상급" in combination:
        pattern = make_all_groups_pattern([k2, k3])

    # ------------------------------------------------------------
    # 단독 키워드형
    # 예:
    # - 의약품명칭
    # - 건기식 오인표현 단독
    # - 현혹표현 단독
    # - 유사과학 용어 단독
    # ------------------------------------------------------------
    elif "단독" in combination or (
        k1 and not k2 and not k3
    ):
        pattern = make_single_pattern(k1)

    # ------------------------------------------------------------
    # 3개 조합형
    # 예:
    # - 비교대상 + 비교어 + 우월표현
    # - 비교대상 + 숫자배수 + 효과/함량
    # ------------------------------------------------------------
    elif k1 and k2 and k3:
        pattern = make_all_groups_pattern([k1, k2, k3])

    # ------------------------------------------------------------
    # 2개 조합형
    # 예:
    # - 질병 + 치료표현
    # - 의약품관련어 + 대체표현
    # - 전문가 주체 + 추천행위
    # ------------------------------------------------------------
    elif k1 and k2:
        pattern = make_all_groups_pattern([k1, k2])

    # ------------------------------------------------------------
    # 그 외
    # ------------------------------------------------------------
    elif k1:
        pattern = make_single_pattern(k1)

    return RulePattern(
        rule_id=rule_id,
        category=category,
        subtype=subtype,
        combination=combination,
        pattern=pattern,
        weight=weight,
        requires_db_check=requires_db_check,
        exception_type=exception_type
    )


# ============================================================
# 5. 엑셀에서 키워드 사전 로드
# ============================================================

def load_rule_patterns_from_excel(
    excel_path: str,
    sheet_name: str = "키워드 사전"
) -> List[RulePattern]:
    """
    엑셀 파일에서 1.1~3.1 세부 유형 행을 읽고 정규표현식을 생성.
    """

    raw = pd.read_excel(excel_path, sheet_name=sheet_name, header=None)

    # 헤더 행 찾기
    header_row_idx = None
    for idx, row in raw.iterrows():
        row_values = [str(v).strip() for v in row.tolist() if not is_empty(v)]
        if "유형" in row_values and "세부 유형" in row_values and "가능한 조합" in row_values:
            header_row_idx = idx
            break

    if header_row_idx is None:
        raise ValueError("헤더 행을 찾지 못했습니다. '유형', '세부 유형', '가능한 조합' 열이 필요합니다.")

    df = pd.read_excel(excel_path, sheet_name=sheet_name, header=header_row_idx)

    # 컬럼명 공백 제거
    df.columns = [str(col).strip() for col in df.columns]

    required_columns = [
        "유형",
        "세부 유형",
        "가능한 조합",
        "키워드사전1",
        "키워드사전2",
        "키워드사전3",
        "위험도 가중치"
    ]

    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"필수 컬럼이 없습니다: {col}")

    # 유형 열이 비어 있는 행은 이전 유형을 이어받음
    df["유형"] = df["유형"].ffill()

    rules: List[RulePattern] = []

    for _, row in df.iterrows():
        subtype = row.get("세부 유형")
        combination = row.get("가능한 조합")

        if is_empty(subtype) or is_empty(combination):
            continue

        # DB 구축 예정처럼 가중치가 없는 본문 설명 행은 제외하되,
        # 예외처리 행은 따로 남김
        if is_empty(row.get("위험도 가중치")) and "예외처리" not in str(subtype):
            continue

        rule_pattern = generate_pattern_for_row(row)

        # 실제 탐지 가능한 패턴이 있거나 예외처리 행이면 추가
        if rule_pattern.pattern or rule_pattern.exception_type:
            rules.append(rule_pattern)

    return rules


# ============================================================
# 6. 예외 패턴 생성
# ============================================================

def build_exception_patterns() -> Dict[str, List[re.Pattern]]:
    """
    현재 스프레드시트의 예외처리 필요 항목을 기반으로 한 기본 예외 패턴.
    추후 데이터 분석 후 보완 가능.
    """

    expert_subject = r"(?:의사|전문의|치과의사|한의사|수의사|약사|한약사|대학교수|교수|의료진)"
    top_claim = r"(?:최초|최고|유일|1위|독보적|최상|no\.1|넘버원|선도적|대표|원조|유일무이|탑클래스|프리미엄)"
    competitor = r"(?:타사|다른\s*제품|기존\s*제품|일반\s*제품|경쟁\s*제품|시중\s*제품|타\s*브랜드|일반\s*영양제|기존\s*방식|타\s*업체)"

    exceptions = {
        # 2.2 전문가 추천/보증/사용 예외:
        # 의사 등이 연구·개발에 직접 참여한 사실만 표시하는 경우
        "expert_participation": [
            re.compile(rf"{expert_subject}.{{0,15}}(?:연구|개발)\s*참여"),
            re.compile(rf"{expert_subject}.{{0,15}}(?:공동\s*개발|직접\s*개발|개발자|연구진)"),
        ],

        # 3.1A 최상급 표현 예외:
        # 조사근거, 조사기관, 조사기간 등이 함께 명시된 경우
        "top_claim_evidence": [
            re.compile(rf"{top_claim}.{{0,25}}(?:조사|기관|근거|출처|자료|보고서|리서치|설문|통계)"),
            re.compile(rf"{top_claim}.{{0,25}}(?:\d{{4}}년|\d{{1,2}}월|\d{{1,2}}일)"),
        ],

        # 3.1B 비교 표현 예외:
        # 비교 기준과 수치 근거가 함께 제시된 경우
        "comparison_evidence": [
            re.compile(rf"{competitor}.{{0,25}}(?:비교\s*기준|시험\s*결과|분석\s*결과|공인\s*성적서|자료|근거)"),
            re.compile(rf"{competitor}.{{0,25}}(?:\d+\.?\d*%|\d+배).{{0,25}}(?:근거|자료|시험|분석|성적서)"),
        ],

        # 수상/선정명 + 연도 예외
        "award_evidence": [
            re.compile(r"(?:수상|선정|인증).{0,20}(?:\d{4}년|\d{4})"),
        ],
    }

    return exceptions


def is_exception_for_rule(sentence: str, rule: RulePattern, exception_patterns: Dict[str, List[re.Pattern]]) -> bool:
    """
    특정 rule 매칭 후 예외 여부 판단.
    """

    text = normalize_text(sentence)

    # 2.2 전문가 추천/보증/사용 예외
    if rule.rule_id in ["2.2A", "2.2B", "2.2C"]:
        return any(p.search(text) for p in exception_patterns["expert_participation"])

    # 3.1A 최초/최고/유일/1위 예외
    if rule.rule_id == "3.1A":
        return any(p.search(text) for p in exception_patterns["top_claim_evidence"]) or \
               any(p.search(text) for p in exception_patterns["award_evidence"])

    # 3.1B 우월 비교 예외
    if rule.rule_id == "3.1B":
        return any(p.search(text) for p in exception_patterns["comparison_evidence"])

    return False


# ============================================================
# 7. 문장 단위 Rule 탐지
# ============================================================

def detect_rule_based(
    sentence: str,
    rules: List[RulePattern],
    threshold: int = 8,
    product_name: Optional[str] = None,
    functional_food_db: Optional[set] = None
) -> Dict[str, Any]:
    """
    문장 하나에 대해 1차 Rule 탐지 수행.

    - 매칭된 rule의 가중치를 합산
    - 총점이 threshold 이상이면 Rule-Positive
    - DB 검증이 필요한 유형은 product_name과 functional_food_db를 통해 처리
    """

    text = normalize_text(sentence)
    exception_patterns = build_exception_patterns()

    matched_rules = []
    total_score = 0

    for rule in rules:
        if not rule.pattern:
            continue

        regex = re.compile(rule.pattern, flags=re.IGNORECASE)

        if not regex.search(text):
            continue

        # 예외처리
        if is_exception_for_rule(text, rule, exception_patterns):
            continue

        # 건강기능식품 DB 검증 필요 유형 처리
        if rule.requires_db_check:
            # DB가 없으면 일단 매칭은 기록하되, 점수 부여 여부는 보류 가능
            if functional_food_db is not None and product_name is not None:
                if product_name in functional_food_db:
                    # DB에 등록된 건강기능식품이면 해당 rule 점수 부여하지 않음
                    continue

        matched_rules.append({
            "rule_id": rule.rule_id,
            "category": rule.category,
            "subtype": rule.subtype,
            "weight": rule.weight,
            "pattern": rule.pattern,
            "requires_db_check": rule.requires_db_check
        })

        total_score += rule.weight

    return {
        "sentence": sentence,
        "score": total_score,
        "rule_positive": total_score >= threshold,
        "matched_rules": matched_rules
    }


# ============================================================
# 8. 테스트 실행 예시
# ============================================================

if __name__ == "__main__":
    excel_path = "/content/drive/MyDrive/Colab Notebooks/rule-based 모델 패턴 정리.xlsx"

    rules = load_rule_patterns_from_excel(excel_path)

    print("생성된 Rule 패턴 수:", len([r for r in rules if r.pattern]))

    # 생성된 정규표현식 확인
    for rule in rules:
        if rule.pattern:
            print("=" * 80)
            print(rule.rule_id, rule.subtype)
            print("조합:", rule.combination)
            print("가중치:", rule.weight)
            print("DB 검증 필요:", rule.requires_db_check)
            print("REGEX:", rule.pattern)

    test_sentences = [
        "당뇨 완치에 도움 되는 제품입니다.",
        "관절염 개선 효과를 느껴보세요.",
        "항암 효과가 있는 건강식품입니다.",
        "혈행개선제처럼 관리할 수 있습니다.",
        "약 안먹어도 괜찮다는 후기가 많습니다.",
        "건강기능식품 수준의 기능성 제품입니다.",
        "면역력 개선에 도움을 주는 제품입니다.",
        "하루만에 체지방 감소 효과를 느껴보세요.",
        "의사가 추천하는 프리미엄 영양제입니다.",
        "의사가 연구개발에 참여한 제품입니다.",
        "세계 최초 포뮬러입니다.",
        "타사보다 흡수율이 3배 높습니다.",
        "다른 제품은 저급 원료를 씁니다."
    ]

    print("\n\n탐지 테스트")
    print("=" * 80)

    for sentence in test_sentences:
        result = detect_rule_based(sentence, rules)

        print("\n문장:", sentence)
        print("점수:", result["score"])
        print("Rule-Positive:", result["rule_positive"])
        print("매칭:", [(m["rule_id"], m["subtype"], m["weight"]) for m in result["matched_rules"]])

생성된 Rule 패턴 수: 22
1.1A A. 질병/증상 예방·치료형
조합: (질병·증상 + 예방/치료표현)
가중치: 10
DB 검증 필요: False
REGEX: (?=.*(?:코로나19|심혈관질환|알츠하이머|소화불량|류마티스|골다공증|우한폐렴|전립선암|고지혈증|수면장애|동맥경화|어지럼증|만성피로|대장암|재채기|관절통|고혈압|건망증|목통증|아토피|관절염|지방간|편두통|코로나|유방암|폐질환|피부염|가려움|불면증|간질환|우울감|인후통|습진|가래|위염|염증|폐암|복통|부종|설사|변비|오한|치매|발열|질염|붓기|피로|당뇨|독감|위암|콧물|기침|장염|두통|암))(?=.*(?:정상화|감소|해소|완화|억제|차단|회복|치료|방지|완치|조절|말끔|경감|효과|개선|예방|확|싹))
1.1B B. 항(抗) 효과형
조합: (항표현 + 질병 + 효과표현)
가중치: 10
DB 검증 필요: False
REGEX: (?:항).{0,8}(?:바이러스|고혈압|염증|당뇨|암|염|균).{0,8}(?:작용|효과|기능|효능)
1.2A A. 질병 연관성 표현형
조합: (질병·질환자 + 연관표현)
가중치: 10
DB 검증 필요: False
REGEX: (?=.*(?:코로나19|심혈관질환|알츠하이머|소화불량|류마티스|골다공증|우한폐렴|전립선암|고지혈증|수면장애|동맥경화|어지럼증|만성피로|대장암|재채기|관절통|고혈압|건망증|목통증|아토피|관절염|지방간|질환자|편두통|코로나|유방암|폐질환|피부염|가려움|불면증|간질환|우울감|인후통|습진|가래|위염|염증|폐암|환자|복통|부종|설사|변비|오한|치매|발열|질염|붓기|피로|당뇨|독감|위암|콧물|기침|장염|두통|암))(?=.*(?:프로그램|솔루션|포뮬러|밸런스|서포트|전용|도움|케어|맞춤|위한|관리|용))
1.3A A. 의약품 유사명칭형
조합: (의약품명칭)
가중치: 10
DB 검증 필요: False
REGEX: (?:식품먹는탈모약|천연수면유도제|긴장완화유도제|항바이러스제|기관지확장제|위산억제제|먹는탈모약|수면유도제|변비치료제|혈행개선제|식욕억제제|식욕억제약|체중감소

In [15]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# 마운트 후, 'rule-based 모델 패턴 정리.xlsx' 파일의 실제 경로를 여기에 입력하세요.
# 예시: excel_path = "/content/drive/MyDrive/rule-based 모델 패턴 정리.xlsx"
#       excel_path = "/content/drive/MyDrive/MyProject/rule-based 모델 패턴 정리.xlsx"
excel_path = "/content/drive/MyDrive/Colab Notebooks/rule-based 모델 패턴 정리.xlsx"

try:
    rules = load_rule_patterns_from_excel(excel_path)

    print("생성된 Rule 패턴 수:", len([r for r in rules if r.pattern]))

    # 생성된 정규표현식 확인 (선택 사항)
    # for rule in rules:
    #     if rule.pattern:
    #         print("=" * 80)
    #         print(rule.rule_id, rule.subtype)
    #         print("조합:", rule.combination)
    #         print("가중치:", rule.weight)
    #         print("DB 검증 필요:", rule.requires_db_check)
    #         print("REGEX:", rule.pattern)

    test_sentences = [
        "당뇨 완치에 도움 되는 제품입니다.",
        "관절염 개선 효과를 느껴보세요.",
        "항암 효과가 있는 건강식품입니다.",
        "혈행개선제처럼 관리할 수 있습니다.",
        "약 안먹어도 괜찮다는 후기가 많습니다.",
        "건강기능식품 수준의 기능성 제품입니다.",
        "면역력 개선에 도움을 주는 제품입니다.",
        "하루만에 체지방 감소 효과를 느껴보세요.",
        "의사가 추천하는 프리미엄 영양제입니다.",
        "의사가 연구개발에 참여한 제품입니다.",
        "세계 최초 포뮬러입니다.",
        "타사보다 흡수율이 3배 높습니다.",
        "다른 제품은 저급 원료를 씁니다."
    ]

    print("\n\n탐지 테스트")
    print("=" * 80)

    for sentence in test_sentences:
        result = detect_rule_based(sentence, rules)

        print("\n문장:", sentence)
        print("점수:", result["score"])
        print("Rule-Positive:", result["rule_positive"])
        print("매칭:", [(m["rule_id"], m["subtype"], m["weight"]) for m in result["matched_rules"]])

except FileNotFoundError:
    print(f"오류: 엑셀 파일을 찾을 수 없습니다. 경로를 확인해 주세요: {excel_path}")
except Exception as e:
    print(f"파일 로드 중 오류 발생: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
생성된 Rule 패턴 수: 22


탐지 테스트

문장: 당뇨 완치에 도움 되는 제품입니다.
점수: 20
Rule-Positive: True
매칭: [('1.1A', 'A. 질병/증상 예방·치료형', 10), ('1.2A', 'A. 질병 연관성 표현형', 10)]

문장: 관절염 개선 효과를 느껴보세요.
점수: 16
Rule-Positive: True
매칭: [('1.1A', 'A. 질병/증상 예방·치료형', 10), ('1.4B', 'B. 기능성 효능 표방형', 6)]

문장: 항암 효과가 있는 건강식품입니다.
점수: 20
Rule-Positive: True
매칭: [('1.1A', 'A. 질병/증상 예방·치료형', 10), ('1.1B', 'B. 항(抗) 효과형', 10)]

문장: 혈행개선제처럼 관리할 수 있습니다.
점수: 24
Rule-Positive: True
매칭: [('1.3A', 'A. 의약품 유사명칭형', 10), ('1.4B', 'B. 기능성 효능 표방형', 6), ('2.1A', 'A. 신체조직 효능 과장형', 8)]

문장: 약 안먹어도 괜찮다는 후기가 많습니다.
점수: 10
Rule-Positive: True
매칭: [('1.3C', 'C. 의약품 대체형', 10)]

문장: 건강기능식품 수준의 기능성 제품입니다.
점수: 8
Rule-Positive: True
매칭: [('1.4A', 'A. 건강기능식품 직접 표방형', 8)]

문장: 면역력 개선에 도움을 주는 제품입니다.
점수: 14
Rule-Positive: True
매칭: [('1.4B', 'B. 기능성 효능 표방형', 6), ('2.1A', 'A. 신체조직 효능 과장형', 8)]

문장: 하루만에 체지방 감소 효과를 느껴보세요.
점수: 24
Rule-P

In [16]:
# 탐지 결과를 저장할 리스트
detection_results = []

for sentence in test_sentences:
    result = detect_rule_based(sentence, rules)
    detection_results.append(result)

# 결과를 데이터프레임으로 변환
results_df = pd.DataFrame(detection_results)

# Rule-Positive가 True인 결과만 필터링
rule_positive_df = results_df[results_df['rule_positive'] == True].copy()

# 'matched_rules' 컬럼을 더 보기 좋게 정리 (선택 사항)
rule_positive_df['matched_rules_summary'] = rule_positive_df['matched_rules'].apply(
    lambda x: [f"{m['rule_id']} ({m['subtype']}) - W:{m['weight']}" for m in x]
)

# 필요한 컬럼만 선택하여 출력
display(rule_positive_df[['sentence', 'score', 'rule_positive', 'matched_rules_summary']])

,sentence,score,rule_positive,matched_rules_summary
0,당뇨 완치에 도움 되는 제품입니다.,20,True,"[1.1A (A. 질병/증상 예방·치료형) - W:10, 1.2A (A. 질병 연관..."
1,관절염 개선 효과를 느껴보세요.,16,True,"[1.1A (A. 질병/증상 예방·치료형) - W:10, 1.4B (B. 기능성 효..."
2,항암 효과가 있는 건강식품입니다.,20,True,"[1.1A (A. 질병/증상 예방·치료형) - W:10, 1.1B (B. 항(抗) ..."
3,혈행개선제처럼 관리할 수 있습니다.,24,True,"[1.3A (A. 의약품 유사명칭형) - W:10, 1.4B (B. 기능성 효능 표..."
4,약 안먹어도 괜찮다는 후기가 많습니다.,10,True,[1.3C (C. 의약품 대체형) - W:10]
5,건강기능식품 수준의 기능성 제품입니다.,8,True,[1.4A (A. 건강기능식품 직접 표방형) - W:8]
6,면역력 개선에 도움을 주는 제품입니다.,14,True,"[1.4B (B. 기능성 효능 표방형) - W:6, 2.1A (A. 신체조직 효능 ..."
7,하루만에 체지방 감소 효과를 느껴보세요.,24,True,"[1.4B (B. 기능성 효능 표방형) - W:6, 2.1A (A. 신체조직 효능 ..."
8,의사가 추천하는 프리미엄 영양제입니다.,12,True,"[2.2A (A. 전문가 추천형) - W:6, 3.1A (A. 최초 / 최고 / 유..."
11,타사보다 흡수율이 3배 높습니다.,10,True,[3.1B (B. 근거 없는 우월 비교) - W:10]


In [18]:
print("\n\n유형별 세부 정규표현식 확인")
print("=" * 80)

for rule in rules:
    if rule.pattern:
        print(f"Rule ID: {rule.rule_id}")
        print(f"  Subtype: {rule.subtype}")
        print(f"  REGEX: {rule.pattern}\n")



유형별 세부 정규표현식 확인
Rule ID: 1.1A
  Subtype: A. 질병/증상 예방·치료형
  REGEX: (?=.*(?:코로나19|심혈관질환|알츠하이머|소화불량|류마티스|골다공증|우한폐렴|전립선암|고지혈증|수면장애|동맥경화|어지럼증|만성피로|대장암|재채기|관절통|고혈압|건망증|목통증|아토피|관절염|지방간|편두통|코로나|유방암|폐질환|피부염|가려움|불면증|간질환|우울감|인후통|습진|가래|위염|염증|폐암|복통|부종|설사|변비|오한|치매|발열|질염|붓기|피로|당뇨|독감|위암|콧물|기침|장염|두통|암))(?=.*(?:정상화|감소|해소|완화|억제|차단|회복|치료|방지|완치|조절|말끔|경감|효과|개선|예방|확|싹))

Rule ID: 1.1B
  Subtype: B. 항(抗) 효과형
  REGEX: (?:항).{0,8}(?:바이러스|고혈압|염증|당뇨|암|염|균).{0,8}(?:작용|효과|기능|효능)

Rule ID: 1.2A
  Subtype: A. 질병 연관성 표현형
  REGEX: (?=.*(?:코로나19|심혈관질환|알츠하이머|소화불량|류마티스|골다공증|우한폐렴|전립선암|고지혈증|수면장애|동맥경화|어지럼증|만성피로|대장암|재채기|관절통|고혈압|건망증|목통증|아토피|관절염|지방간|질환자|편두통|코로나|유방암|폐질환|피부염|가려움|불면증|간질환|우울감|인후통|습진|가래|위염|염증|폐암|환자|복통|부종|설사|변비|오한|치매|발열|질염|붓기|피로|당뇨|독감|위암|콧물|기침|장염|두통|암))(?=.*(?:프로그램|솔루션|포뮬러|밸런스|서포트|전용|도움|케어|맞춤|위한|관리|용))

Rule ID: 1.3A
  Subtype: A. 의약품 유사명칭형
  REGEX: (?:식품먹는탈모약|천연수면유도제|긴장완화유도제|항바이러스제|기관지확장제|위산억제제|먹는탈모약|수면유도제|변비치료제|혈행개선제|식욕억제제|식욕억제약|체중감소약|다이어트약|면역억제제|숙변제거제|고혈압약|암억제제|장청소약|항염제|항암제|소화제|당뇨약|항생제|위장약|소염제|수면제|변비약|혈압약|탈모약|진통

In [20]:
regex_patterns_data = []
for rule in rules:
    if rule.pattern:
        regex_patterns_data.append({
            'rule_id': rule.rule_id,
            'subtype': rule.subtype,
            'regex_pattern': rule.pattern
        })

regex_df = pd.DataFrame(regex_patterns_data)

# CSV 파일로 저장
output_regex_csv_path = "all_rule_regex_patterns.csv"
regex_df.to_csv(output_regex_csv_path, index=False, encoding='utf-8-sig')

print(f"모든 정규표현식 패턴이 '{output_regex_csv_path}' 파일로 저장되었습니다.")
display(regex_df.head())

모든 정규표현식 패턴이 'all_rule_regex_patterns.csv' 파일로 저장되었습니다.


,rule_id,subtype,regex_pattern
0,1.1A,A. 질병/증상 예방·치료형,(?=.*(?:코로나19|심혈관질환|알츠하이머|소화불량|류마티스|골다공증|우한폐렴|...
1,1.1B,B. 항(抗) 효과형,"(?:항).{0,8}(?:바이러스|고혈압|염증|당뇨|암|염|균).{0,8}(?:작용..."
2,1.2A,A. 질병 연관성 표현형,(?=.*(?:코로나19|심혈관질환|알츠하이머|소화불량|류마티스|골다공증|우한폐렴|...
3,1.3A,A. 의약품 유사명칭형,(?:식품먹는탈모약|천연수면유도제|긴장완화유도제|항바이러스제|기관지확장제|위산억제제...
4,1.3B,B. 한약처방 조합형,(?:십전대보탕|팔미지황탕|녹용대보탕|육미지황탕|익수영진|녹용대보|궁귀교애|육미지황...
